In [ ]:
import random
import time
from dataclasses import dataclass
from typing import Optional


# =========================================================
# ЗАДАЧА 1: Сбалансированное дерево поиска
# Реализовано: AVL-дерево (самобалансировка по высоте)
# =========================================================

@dataclass
class AVLNode:
    key: int
    height: int = 1
    count: int = 1                # чтобы корректно обрабатывать дубликаты
    left: Optional["AVLNode"] = None
    right: Optional["AVLNode"] = None


class AVLTree:
    """
    AVL-дерево: вставка и поиск за O(log n) в среднем/худшем.
    Баланс поддерживается поворотами (LL, RR, LR, RL).
    """
    def __init__(self):
        self.root: Optional[AVLNode] = None

    # -------------------------
    # ВСПОМОГАТЕЛЬНЫЕ МЕТОДЫ
    # -------------------------

    def _h(self, node: Optional[AVLNode]) -> int:
        """Высота узла (0 если None)."""
        return node.height if node else 0

    def _update_height(self, node: AVLNode) -> None:
        """Пересчитать высоту узла на основе высот детей."""
        node.height = 1 + max(self._h(node.left), self._h(node.right))

    def _balance_factor(self, node: AVLNode) -> int:
        """Фактор баланса: height(left) - height(right)."""
        return self._h(node.left) - self._h(node.right)

    # -------------------------
    # ПОВОРОТЫ
    # -------------------------

    def _rotate_right(self, y: AVLNode) -> AVLNode:
        """
        Правый поворот вокруг y:
              y              x
             / \            / \
            x  T3   =>     T1  y
           / \                / \
          T1 T2              T2 T3
        """
        x = y.left
        assert x is not None  # для поворота левый ребёнок должен существовать

        T2 = x.right
        x.right = y
        y.left = T2

        self._update_height(y)
        self._update_height(x)
        return x

    def _rotate_left(self, x: AVLNode) -> AVLNode:
        """
        Левый поворот вокруг x:
            x                 y
           / \               / \\
          T1  y     =>      x  T3
             / \           / \\
            T2 T3         T1 T2
        """
        y = x.right
        assert y is not None  # для поворота правый ребёнок должен существовать

        T2 = y.left
        y.left = x
        x.right = T2

        self._update_height(x)
        self._update_height(y)
        return y

    def _rebalance(self, node: AVLNode) -> AVLNode:
        """
        Восстановить баланс узла node после вставки:
        - LL: правый поворот
        - RR: левый поворот
        - LR: левый поворот в левом ребёнке, затем правый поворот
        - RL: правый поворот в правом ребёнке, затем левый поворот
        """
        bf = self._balance_factor(node)

        # Случай Left-Heavy
        if bf > 1:
            # LR: если левый ребёнок "наклонён" вправо
            if node.left and self._balance_factor(node.left) < 0:
                node.left = self._rotate_left(node.left)
            return self._rotate_right(node)

        # Случай Right-Heavy
        if bf < -1:
            # RL: если правый ребёнок "наклонён" влево
            if node.right and self._balance_factor(node.right) > 0:
                node.right = self._rotate_right(node.right)
            return self._rotate_left(node)

        return node

    # -------------------------
    # ВСТАВКА
    # -------------------------

    def insert(self, key: int) -> None:
        """Публичная вставка."""
        self.root = self._insert(self.root, key)

    def _insert(self, node: Optional[AVLNode], key: int) -> AVLNode:
        """Рекурсивная вставка с балансировкой."""
        if node is None:
            return AVLNode(key=key)

        if key == node.key:
            node.count += 1  # дубликат
            return node
        elif key < node.key:
            node.left = self._insert(node.left, key)
        else:
            node.right = self._insert(node.right, key)

        # обновляем высоту и балансируем
        self._update_height(node)
        return self._rebalance(node)

    # -------------------------
    # ПОИСК
    # -------------------------

    def contains(self, key: int) -> bool:
        """Проверка наличия ключа."""
        cur = self.root
        while cur is not None:
            if key == cur.key:
                return cur.count > 0
            cur = cur.left if key < cur.key else cur.right
        return False


def benchmark_avl_vs_list(
    n_insert: int = 30_000,
    n_query: int = 30_000,
    low: int = -1_000_000,
    high: int = 1_000_000,
    seed: int = 42
) -> None:
    """
    Основная программа для замеров:
    1) генерируем n_insert чисел -> пишем в list и AVL
    2) генерируем n_query чисел -> проверяем наличие в list и AVL, считаем совпадения
    3) повторяем проверку только в AVL и сравниваем времена

    Важно:
    - проверка `x in list` линейная по размеру списка, поэтому для Python
      n_insert=n_query=100_000 может выполняться слишком долго.
    - начни с 30_000 и увеличивай по необходимости.
    """
    rnd = random.Random(seed)

    # 1) Подготовка структур
    arr = []
    tree = AVLTree()

    # --- замер вставки ---
    t0 = time.perf_counter()
    for _ in range(n_insert):
        x = rnd.randint(low, high)
        arr.append(x)
        tree.insert(x)
    t1 = time.perf_counter()
    insert_time = t1 - t0

    # 2) Проверка наличия: list + tree
    hits_list = 0
    hits_tree = 0

    t2 = time.perf_counter()
    queries = [rnd.randint(low, high) for _ in range(n_query)]
    for q in queries:
        if q in arr:          # линейный поиск по списку
            hits_list += 1
        if tree.contains(q):  # логарифмический поиск в AVL
            hits_tree += 1
    t3 = time.perf_counter()
    both_check_time = t3 - t2

    # ответы должны совпасть (оба заполнены одними и теми же числами)
    print("=== РЕЗУЛЬТАТЫ (list + AVL) ===")
    print(f"Вставка n={n_insert}: {insert_time:.4f} сек")
    print(f"Проверка n={n_query} (list+AVL): {both_check_time:.4f} сек")
    print(f"Совпадений в list: {hits_list}")
    print(f"Совпадений в AVL : {hits_tree}")
    print("Совпадают ли ответы:", "ДА" if hits_list == hits_tree else "НЕТ")

    # 3) Проверка только AVL
    hits_tree_only = 0
    t4 = time.perf_counter()
    for q in queries:
        if tree.contains(q):
            hits_tree_only += 1
    t5 = time.perf_counter()
    tree_only_time = t5 - t4

    print("\n=== ТОЛЬКО AVL ===")
    print(f"Проверка n={n_query} (только AVL): {tree_only_time:.4f} сек")
    print(f"Совпадений в AVL: {hits_tree_only}")

    print("\n=== СРАВНЕНИЕ ВРЕМЕНИ ПРОВЕРОК ===")
    print(f"list+AVL : {both_check_time:.4f} сек")
    print(f"только AVL: {tree_only_time:.4f} сек")
    if tree_only_time > 0:
        print(f"Ускорение (примерно): {both_check_time / tree_only_time:.2f}x")


# Запуск (можно менять n_insert/n_query):
benchmark_avl_vs_list(n_insert=30_000, n_query=30_000)

<>:50: SyntaxWarning: invalid escape sequence '\ '
<>:70: SyntaxWarning: invalid escape sequence '\ '
<>:50: SyntaxWarning: invalid escape sequence '\ '
<>:70: SyntaxWarning: invalid escape sequence '\ '
C:\Users\zerok\AppData\Local\Temp\ipykernel_30124\980292203.py:50: SyntaxWarning: invalid escape sequence '\ '
  """
C:\Users\zerok\AppData\Local\Temp\ipykernel_30124\980292203.py:70: SyntaxWarning: invalid escape sequence '\ '
  """


=== РЕЗУЛЬТАТЫ (list + AVL) ===
Вставка n=30000: 0.3830 сек
Проверка n=30000 (list+AVL): 4.6733 сек
Совпадений в list: 413
Совпадений в AVL : 413
Совпадают ли ответы: ДА

=== ТОЛЬКО AVL ===
Проверка n=30000 (только AVL): 0.0377 сек
Совпадений в AVL: 413

=== СРАВНЕНИЕ ВРЕМЕНИ ПРОВЕРОК ===
list+AVL : 4.6733 сек
только AVL: 0.0377 сек
Ускорение (примерно): 123.89x
